In [ ]:
!pip install notion-client python-dotenv

Notion integration part

In [ ]:
from google.colab import userdata

NOTION_TOKEN = userdata.get("NOTION_TOKEN")
NOTION_PARENT_PAGE_ID = userdata.get("NOTION_PARENT_PAGE_ID")

In [ ]:
from google.colab import userdata
from notion_client import Client

notion = Client(auth=userdata.get("NOTION_TOKEN"))
PARENT_PAGE_ID = userdata.get("NOTION_PARENT_PAGE_ID")

In [ ]:
import re

def parse_plain_text_to_blocks(text: str) -> tuple:
    """
    Parses Gemini's plain-text structured output into:
    - title (str): extracted from the first heading line
    - blocks (list): list of Notion block dicts

    Detects:
      - Headings: lines ending with ':'
      - Bullets:  lines starting with •, -, *
      - Numbered: lines starting with 1. 2. etc.
      - Paragraphs: everything else
    """
    lines = text.strip().split("\n")
    blocks = []
    title = None
    i = 0

    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        # Skip blank lines
        if not stripped:
            i += 1
            continue

        # --- Heading: line ends with ':' ---
        if stripped.endswith(":") and len(stripped) > 1:
            heading_text = stripped[:-1].strip()  # remove the colon

            # First heading becomes the page title, not a block
            if title is None:
                title = heading_text
            else:
                blocks.append(_make_heading(heading_text, level=2))

            i += 1
            continue

        # --- Bullet point: starts with •, -, or * ---
        if stripped.startswith(("-", "*","+)","->")):
            # Strip the bullet symbol and any whitespace after it
            bullet_text = stripped.lstrip("•-* +)>").strip()
            blocks.append(_make_bullet(bullet_text))
            i += 1
            continue

        # --- Numbered list: starts with "1." "2." etc. ---
        if re.match(r"^\d+[\.\)]\s", stripped):
            item_text = re.sub(r"^\d+[\.\)]\s+", "", stripped).strip()
            blocks.append(_make_numbered(item_text))
            i += 1
            continue

        # --- Paragraph: collect consecutive non-special lines ---
        para_lines = []
        while i < len(lines):
            current = lines[i].strip()

            # Stop collecting if we hit a blank line or a special line
            if not current:
                break
            if _is_special_line(current):
                break

            para_lines.append(current)
            i += 1

        if para_lines:
            paragraph_text = " ".join(para_lines)
            blocks.append(_make_paragraph(paragraph_text))
        continue  # don't increment i again, already done in inner loop

    # Fallback title if Gemini gave no heading at all
    if title is None:
        from datetime import datetime
        title = f"Note — {datetime.now().strftime('%Y-%m-%d %H:%M')}"

    return title, blocks


def _is_special_line(line: str) -> bool:
    """Returns True for lines that should NOT be merged into a paragraph."""
    return (
        line.endswith(":")                    or  # heading
        line.startswith(("-", "*","+)","->"))      or  # bullet
        re.match(r"^\d+[\.\)]\s", line)           # numbered
    )


# --- Block factory functions ---

def _make_heading(text: str, level: int = 2) -> dict:
    block_type = f"heading_{level}"
    return {
        "object": "block",
        "type": block_type,
        block_type: {
            "rich_text": [{"type": "text", "text": {"content": text}}]
        }
    }

def _make_paragraph(text: str) -> dict:
    return {
        "object": "block",
        "type": "paragraph",
        "paragraph": {
            "rich_text": [{"type": "text", "text": {"content": text}}]
        }
    }

def _make_bullet(text: str) -> dict:
    return {
        "object": "block",
        "type": "bulleted_list_item",
        "bulleted_list_item": {
            "rich_text": [{"type": "text", "text": {"content": text}}]
        }
    }

def _make_numbered(text: str) -> dict:
    return {
        "object": "block",
        "type": "numbered_list_item",
        "numbered_list_item": {
            "rich_text": [{"type": "text", "text": {"content": text}}]
        }
    }

In [ ]:
import os
from datetime import datetime
from notion_client import Client
from dotenv import load_dotenv


def create_notion_page(gemini_text: str) -> str:
    """
    Takes your Gemini plain-text output string,
    creates a formatted Notion page, returns the URL.
    """

    # Step 1: Parse the text into a title + list of blocks
    title, content_blocks = parse_plain_text_to_blocks(gemini_text)

    # Step 2: Build header blocks (metadata callout + divider)
    timestamp = datetime.now().strftime("%Y-%m-%d at %H:%M")
    header_blocks = [
        {
            "object": "block",
            "type": "callout",
            "callout": {
                "rich_text": [{
                    "type": "text",
                    "text": {"content": f"Auto-imported on {timestamp}"}
                }],
                "icon": {"emoji": "📷"}
            }
        },
        {
            "object": "block",
            "type": "divider",
            "divider": {}
        }
    ]

    # Step 3: Combine header + content
    all_blocks = header_blocks + content_blocks

    # Step 4: Create the page with first batch (max 100 blocks)
    first_batch = all_blocks[:100]

    response = notion.pages.create(
        parent={"page_id": PARENT_PAGE_ID},
        properties={
            "title": {
                "title": [
                    {"type": "text", "text": {"content": title}}
                ]
            }
        },
        children=first_batch
    )

    page_id = response["id"]

    # Step 5: Append remaining blocks if note exceeds 100 blocks
    remaining = all_blocks[100:]
    while remaining:
        batch = remaining[:100]
        remaining = remaining[100:]
        notion.blocks.children.append(
            block_id=page_id,
            children=batch
        )

    # Step 6: Return the page URL
    page_url = f"https://www.notion.so/{page_id.replace('-', '')}"
    print(f"✅ Notion page created: {page_url}")
    return page_url

Example

In [ ]:

gemini_output = """ Tesseract:
Ican ocR engine That uses Deep learning
and a neural network architecture called
(sTu(Long short-Term Memory)
PiPeline:
+) Lay out Analysis: figumes out where the
 lines and words are
+) Feature Extraction: breaks the image into
tinymath paterns
+) Neural Network
+) Language Model: checks adict tosee if the
wodexists  TPotto h Tt """

page_url = create_notion_page(gemini_output)
print(f"Pipeline complete. Note saved: {page_url}")

✅ Notion page created: https://www.notion.so/350e54104f86819aba08d4d2535ad4d0
Pipeline complete. Note saved: https://www.notion.so/350e54104f86819aba08d4d2535ad4d0
